In [14]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import polars as pl
from PIL import Image
from IPython.display import display

In [3]:
# load cpi embeddings
parquet_path = '/Users/josephko/research/cpi-thermo/data/raw/SSL-Model-v3/cpi3m_campaign_cls_head_features_compressed.parquet'
df = pl.read_parquet(parquet_path)
unique_campaigns = df["campaign"].unique()
print(f"{len(unique_campaigns.to_list())} Unique campaigns:", unique_campaigns.to_list())

12 Unique campaigns: ['ATTREX', 'AIRS_II', 'ICE_L', 'MC3E', 'MIDCIX', 'ARM', 'MPACE', 'IPHEX', 'ISDAC', 'CRYSTAL_FACE_NASA', 'MACPEX', 'CRYSTAL_FACE_UND']


In [4]:
df.head()

campaign_file_id,cls_features,campaign,filename,head_features
str,list[f32],str,str,list[f32]
"""AIRS_II/1114-115708_753_18.png""","[-1.573681, -1.34678, … 1.054788]","""AIRS_II""","""1114-115708_753_18.png""","[-0.174372, 0.220068, … 0.195601]"
"""AIRS_II/1114-115708_753_25.png""","[-0.664088, -0.620979, … 1.610363]","""AIRS_II""","""1114-115708_753_25.png""","[0.150885, 0.086997, … 0.107423]"
"""AIRS_II/1114-115708_753_35.png""","[1.181937, -2.416504, … 0.973358]","""AIRS_II""","""1114-115708_753_35.png""","[0.008603, 0.284243, … 0.01234]"
"""AIRS_II/1114-115708_753_4.png""","[1.407759, -2.779922, … 0.635076]","""AIRS_II""","""1114-115708_753_4.png""","[-0.098592, 0.049973, … 0.028704]"
"""AIRS_II/1114-115745_814_121.pn…","[0.783167, -2.249387, … -0.949386]","""AIRS_II""","""1114-115745_814_121.png""","[-0.042272, -0.122378, … -0.023314]"


Parse date and time from filename and add as column

In [5]:
# Create unified datetime column handling both filename formats
df = df.with_columns(

    pl.when(
        pl.col("filename").str.contains(r"^\d{4}-")
    )

    # AIRS_II legacy format:
    # 1114-155011_394_15.png
    # -> 2003_1114_155011
    .then(

        (
            pl.lit("2003_")
            + pl.col("filename")
                .str.extract(r"^(\d{4}-\d{6})")
                .str.replace("-", "_")
        )

        .str.strptime(
            pl.Datetime,
            format="%Y_%m%d_%H%M%S",
            strict=False,
        )

    )

    # Standard format:
    # 2004_1017_212512_578_43.png
    .otherwise(

        pl.col("filename")
        .str.extract(r"(\d{4}_\d{4}_\d{6})")
        .str.strptime(
            pl.Datetime,
            format="%Y_%m%d_%H%M%S",
            strict=False,
        )

    )

    .alias("datetime")

)

# Optional check
print(
    df.select(["filename", "datetime"])
    .head(20)
)

shape: (20, 2)
┌─────────────────────────┬─────────────────────┐
│ filename                ┆ datetime            │
│ ---                     ┆ ---                 │
│ str                     ┆ datetime[μs]        │
╞═════════════════════════╪═════════════════════╡
│ 1114-115708_753_18.png  ┆ 2003-11-14 11:57:08 │
│ 1114-115708_753_25.png  ┆ 2003-11-14 11:57:08 │
│ 1114-115708_753_35.png  ┆ 2003-11-14 11:57:08 │
│ 1114-115708_753_4.png   ┆ 2003-11-14 11:57:08 │
│ 1114-115745_814_121.png ┆ 2003-11-14 11:57:45 │
│ …                       ┆ …                   │
│ 1114-115745_814_199.png ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_2.png   ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_200.png ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_215.png ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_226.png ┆ 2003-11-14 11:57:45 │
└─────────────────────────┴─────────────────────┘


In [6]:
df.head()

campaign_file_id,cls_features,campaign,filename,head_features,datetime
str,list[f32],str,str,list[f32],datetime[μs]
"""AIRS_II/1114-115708_753_18.png""","[-1.573681, -1.34678, … 1.054788]","""AIRS_II""","""1114-115708_753_18.png""","[-0.174372, 0.220068, … 0.195601]",2003-11-14 11:57:08
"""AIRS_II/1114-115708_753_25.png""","[-0.664088, -0.620979, … 1.610363]","""AIRS_II""","""1114-115708_753_25.png""","[0.150885, 0.086997, … 0.107423]",2003-11-14 11:57:08
"""AIRS_II/1114-115708_753_35.png""","[1.181937, -2.416504, … 0.973358]","""AIRS_II""","""1114-115708_753_35.png""","[0.008603, 0.284243, … 0.01234]",2003-11-14 11:57:08
"""AIRS_II/1114-115708_753_4.png""","[1.407759, -2.779922, … 0.635076]","""AIRS_II""","""1114-115708_753_4.png""","[-0.098592, 0.049973, … 0.028704]",2003-11-14 11:57:08
"""AIRS_II/1114-115745_814_121.pn…","[0.783167, -2.249387, … -0.949386]","""AIRS_II""","""1114-115745_814_121.png""","[-0.042272, -0.122378, … -0.023314]",2003-11-14 11:57:45


In [13]:
# keep grouping in Polars, then convert the small result to pandas for formatting
campaign_time_range = (
    df.group_by("campaign")
      .agg(
          pl.col("datetime").min().alias("start_time"),
          pl.col("datetime").max().alias("end_time"),
      )
)

pdf = campaign_time_range.to_pandas()
pdf["start_time"] = pd.to_datetime(pdf["start_time"])
pdf["end_time"] = pd.to_datetime(pdf["end_time"])
pdf["available_duration"] = pdf["end_time"] - pdf["start_time"]

# build formatted duration string
td = pdf["available_duration"]
days = td.dt.days
hours = (td.dt.seconds // 3600).astype(int)
minutes = ((td.dt.seconds % 3600) // 60).astype(int)
seconds = (td.dt.seconds % 60).astype(int)

pdf["duration"] = (
    days.astype(str)
    + " days "
    + hours.astype(str).str.zfill(2)
    + ":"
    + minutes.astype(str).str.zfill(2)
    + ":"
    + seconds.astype(str).str.zfill(2)
)

pdf["start_time"] = pdf["start_time"].dt.strftime("%Y-%m-%d %H:%M:%S")
pdf["end_time"] = pdf["end_time"].dt.strftime("%Y-%m-%d %H:%M:%S")

display(pdf[["campaign","start_time","end_time","duration"]])

,campaign,start_time,end_time,duration
0,MPACE,2004-09-30 01:01:20,2004-10-22 00:51:01,21 days 23:49:41
1,ARM,2000-03-06 20:20:27,2000-03-20 15:36:38,13 days 19:16:11
2,CRYSTAL_FACE_NASA,2002-07-09 15:32:38,2002-07-29 20:12:55,20 days 04:40:17
3,IPHEX,2014-04-22 21:04:16,2014-06-13 16:43:37,51 days 19:39:21
4,ATTREX,2014-03-04 17:41:32,2014-03-05 06:09:23,0 days 12:27:51
5,MIDCIX,2004-04-19 17:03:29,2004-05-06 20:43:29,17 days 03:40:00
6,AIRS_II,2003-11-14 11:57:08,2003-11-19 20:13:33,5 days 08:16:25
7,CRYSTAL_FACE_UND,2002-07-03 17:30:06,2002-07-29 21:16:18,26 days 03:46:12
8,MACPEX,2011-04-03 17:11:10,2011-04-26 17:31:35,23 days 00:20:25
9,MC3E,2011-05-23 16:35:05,2011-06-02 12:31:28,9 days 19:56:23


Save CPI dates as CSV for reference. Pretty much the same df except the embeddings.

In [15]:
out_path = '/Users/josephko/research/cpi-thermo/data/raw/cpi_embeddings_timestamps.csv'
df_dates = df.select(["campaign", "filename", "datetime"]).to_pandas()
df_dates["datetime"] = pd.to_datetime(df_dates["datetime"]).dt.strftime("%Y-%m-%d %H:%M:%S")
df_dates.to_csv(out_path, index=False)